# Visualizing intermediate activations

What a ConvNet sees at each depth, on one photograph — from edges at the bottom to abstractions with no visual meaning at the top.

**Runs on:** CPU — about 2 minutes &nbsp;·&nbsp; **Slides:** [Chapter 10 — Interpreting What ConvNets Learn](../../../course-web-slides/ch10/index.html) &nbsp;·&nbsp; **Section:** 01 — Visualizing intermediate activations

---

## A model, and an image it has never seen

In [ ]:
import keras
import numpy as np
import matplotlib.pyplot as plt

# Use the model trained in chapter 8, or any convnet you have.
model = keras.models.load_model("convnet_from_scratch_with_augmentation.keras")
model.summary()

In [ ]:
img_path = keras.utils.get_file(
    fname="cat.jpg",
    origin="https://img-datasets.s3.amazonaws.com/cat.jpg")

def get_img_array(path, target_size):
    img = keras.utils.load_img(path, target_size=target_size)
    array = keras.utils.img_to_array(img)
    return np.expand_dims(array, axis=0)

img_tensor = get_img_array(img_path, target_size=(180, 180))
plt.imshow(img_tensor[0].astype("uint8")); plt.axis("off"); plt.show()

## A model that returns every layer's output

This is the capability chapter 7 said subclassing throws away. A Functional model is a **graph**, so you can ask any node for its value.

In [ ]:
layer_outputs = []
layer_names = []
for layer in model.layers:
    if isinstance(layer, (keras.layers.Conv2D, keras.layers.MaxPooling2D)):
        layer_outputs.append(layer.output)
        layer_names.append(layer.name)

activation_model = keras.Model(inputs=model.input, outputs=layer_outputs)
activations = activation_model.predict(img_tensor, verbose=0)

for name, act in zip(layer_names, activations):
    print(f"{name:20s} {act.shape}")

## The first layer

In [ ]:
first = activations[0]
fig, axes = plt.subplots(4, 8, figsize=(12, 6.4))
for ax, i in zip(axes.ravel(), range(min(32, first.shape[-1]))):
    ax.imshow(first[0, :, :, i], cmap="viridis")
    ax.set_title(f"ch {i}", fontsize=7); ax.axis("off")
plt.suptitle("First convolution layer — 32 channels", y=1.0)
plt.tight_layout(); plt.show()

Edge detectors, colour blobs, and one or two channels that are **almost blank** — filters that never learned anything useful. That is normal, and worth noticing: not every filter earns its place.

## Every layer, side by side

In [ ]:
images_per_row = 16
for layer_name, layer_activation in zip(layer_names, activations):
    n_features = layer_activation.shape[-1]
    size = layer_activation.shape[1]
    n_cols = n_features // images_per_row
    if n_cols < 1:
        continue
    display_grid = np.zeros(((size + 1) * n_cols - 1,
                             images_per_row * (size + 1) - 1))
    for col in range(n_cols):
        for row in range(images_per_row):
            channel_index = col * images_per_row + row
            channel_image = layer_activation[0, :, :, channel_index].copy()
            if channel_image.sum() != 0:
                channel_image -= channel_image.mean()
                channel_image /= (channel_image.std() + 1e-7)
                channel_image *= 64
                channel_image += 128
            channel_image = np.clip(channel_image, 0, 255).astype("uint8")
            display_grid[col * (size + 1): (col + 1) * size + col,
                         row * (size + 1): (row + 1) * size + row] = channel_image
    scale = 1. / size
    plt.figure(figsize=(scale * display_grid.shape[1],
                        scale * display_grid.shape[0]))
    plt.title(layer_name); plt.grid(False); plt.axis("off")
    plt.imshow(display_grid, aspect="auto", cmap="viridis")
plt.show()

## Three things to read off these pictures

**The first layer is almost a collection of edge detectors.** It retains nearly all the information in the original image — you could reconstruct the cat from it.

**Deeper layers become abstract and less visually interpretable.** They start encoding *cat ear*, *whisker texture* — concepts rather than pixels.

**Sparsity increases with depth.** More and more channels are blank for any given input, because a filter that detects a specific thing is silent on images that do not contain it.

## Measuring the sparsity claim

In [ ]:
fracs = []
for name, act in zip(layer_names, activations):
    blank = (act[0].reshape(-1, act.shape[-1]).max(axis=0) == 0).mean()
    fracs.append(blank)
    print(f"{name:20s} {blank:5.1%} of channels are entirely zero")

plt.figure(figsize=(7, 3.6))
plt.plot(range(len(fracs)), fracs, "o-")
plt.xticks(range(len(fracs)), layer_names, rotation=45, ha="right")
plt.ylabel("fraction of dead channels"); plt.tight_layout()
plt.title("Sparsity increases with depth")
plt.show()

This is what the chapter means by a network **distilling** its input: information about *what the image is* survives, while information about *what it looks like* is progressively discarded. A ConvNet is an information funnel, and the funnel is deliberate.

---

## What to take away

- A Functional model lets you build a second model that outputs any intermediate layer.
- The first layer is nearly a lossless edge detector; deeper layers encode concepts.
- **Sparsity rises with depth** — specific detectors are silent on most inputs.
- The network discards appearance and keeps identity. That is the point, not a defect.